# Train a small AdjMatSeer on OpenBabel-labelled bonds

Data: `./bond_pairs/shard*.pt` from `generate_obabel_pairs.ipynb`

- `elements` (int8, DIMENSION) — atomic numbers, canonical order
- `coords` (f16, DIMENSION, 3) — canonical order, Å
- `conn` (int8, D, D) — RDKit `DetermineConnectivity` guess → **model INPUT**
- `target` (int8, D, D) — OpenBabel bond orders 0–4 → **model TARGET**
- `n_atoms` (int16)

Inputs are rebuilt to match `prepare_adj_mat_seer_input` exactly: `dist_mat + I`, and
binary `(conn + I)` clamped to 1. Nothing here may use `target` to build an input.

Selection metric is molecule **VALIDITY** (RDKit sanitization of the reconstructed graph),
not the loss — that is the number the README quotes as 48% (ML) vs 93% (OpenBabel).

The 2048-wide baseline is ~21.8 M params / 83 MB; 512 → ~1.6 M / 6 MB, 256 → ~0.5 M / 2 MB.
Seer compute is only ~5% of an 8-NFE sample, so expect a packaging win, not a speed win.

Set `N_HIDDEN` (512, then 256) and optionally `BASELINE` to score the production seer on the
same val split. Only its `valid` number is comparable if labels are Kekulé: it was trained with
aromatic = class 4.

In [ ]:
"""Train a small AdjMatSeer. Config is the block below — rerun with N_HIDDEN=256 after 512."""
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from rdkit import Chem, RDLogger
from torch.optim import AdamW
from tqdm import tqdm

try:
    from src.mlconfgen.adj_mat_seer import AdjMatSeer
    from src.mlconfgen.utils.common import bond_type_dict
    from src.mlconfgen.utils.config import DIMENSION, NUM_BOND_TYPES
except ImportError:
    from mlconfgen.adj_mat_seer import AdjMatSeer
    from mlconfgen.utils.common import bond_type_dict
    from mlconfgen.utils.config import DIMENSION, NUM_BOND_TYPES

RDLogger.DisableLog("rdApp.*")

# --------------------- config ---------------------
DATA_DIR = Path("./bond_pairs")
OUT_DIR = Path("./checkpoints_seer")
N_HIDDEN = 512              # then 256
EPOCHS = 30
BATCH = 128
LR = 3e-4
VAL_FRAC = 0.02
MAX_SHARDS = None           # int to cap RAM
WEIGHT_CAP = 25.0           # cap on inverse-frequency class weight
N_VALIDITY = 2000           # molecules per eval for the validity metric
BASELINE = Path("./adj_mat_seer_chembl_15_39.pt")  # None to skip; 2048-wide production seer
PATIENCE = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / f"train_seer_{N_HIDDEN}.log"
torch.manual_seed(SEED)


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")


# --------------------- data ---------------------
class BondData:
    """All shards in RAM (int8/f16 -> ~3.6 kB per molecule)."""

    def __init__(self, data_dir: Path, max_shards: int | None):
        shards = sorted(data_dir.glob("shard*.pt"))
        if not shards:
            raise FileNotFoundError(f"no shard*.pt in {data_dir}")
        if max_shards:
            shards = shards[:max_shards]
        packs = [torch.load(p, map_location="cpu", weights_only=False) for p in shards]
        self.elements = torch.cat([p["elements"] for p in packs])
        self.coords = torch.cat([p["coords"] for p in packs])
        self.conn = torch.cat([p["conn"] for p in packs])
        self.target = torch.cat([p["target"] for p in packs])
        self.n_atoms = torch.cat([p["n_atoms"] for p in packs])
        self.aromatic_mode = packs[0].get("aromatic_mode", "?")
        nbytes = sum(
            t.numel() * t.element_size()
            for t in (self.elements, self.coords, self.conn, self.target)
        )
        log(
            f"loaded {len(shards)} shards  n={self.n_atoms.shape[0]} "
            f"arom={self.aromatic_mode}  ram={nbytes / 1e9:.2f} GB"
        )

    def __len__(self):
        return self.n_atoms.shape[0]


def build_inputs(d: BondData, idx: torch.Tensor, device: str):
    """-> elements(long), dist_mat(float), adj_in(float), target(long), n_atoms(long)"""
    elements = d.elements[idx].to(device=device, dtype=torch.long)
    coords = d.coords[idx].to(device=device, dtype=torch.float32)
    conn = d.conn[idx].to(device=device, dtype=torch.float32)
    target = d.target[idx].to(device=device, dtype=torch.long)
    n_atoms = d.n_atoms[idx].to(device=device, dtype=torch.long)

    eye = torch.eye(DIMENSION, device=device)
    valid = (torch.arange(DIMENSION, device=device)[None, :] < n_atoms[:, None]).float()
    pair_valid = valid[:, :, None] * valid[:, None, :]
    dist = torch.cdist(coords, coords) * pair_valid + eye
    adj_in = (conn + eye).clamp(max=1.0)
    return elements, dist, adj_in, target, n_atoms


def pair_mask(n_atoms: torch.Tensor, device: str) -> torch.Tensor:
    """Upper triangle of the real n x n block; the model output is symmetric."""
    ar = torch.arange(DIMENSION, device=device)
    valid = ar[None, :] < n_atoms[:, None]
    both = valid[:, :, None] & valid[:, None, :]
    return both & (ar[None, :, None] < ar[None, None, :])


def class_weights(d: BondData, n_sample: int, cap: float, device: str) -> torch.Tensor:
    """Inverse sqrt-frequency, capped. ~94% of pairs are 'no bond'."""
    idx = torch.arange(min(n_sample, len(d)))
    tgt = d.target[idx].to(torch.long)
    na = d.n_atoms[idx].to(torch.long)
    m = pair_mask(na, "cpu")
    vals = tgt[m]
    counts = torch.bincount(vals, minlength=NUM_BOND_TYPES).float().clamp_min(1.0)
    w = (counts.sum() / counts).sqrt()
    w = (w / w[0]).clamp(max=cap)
    return w.to(device)


def mol_is_valid(elements: np.ndarray, orders: np.ndarray, n: int) -> bool:
    m = Chem.RWMol()
    for z in elements[:n]:
        m.AddAtom(Chem.Atom(int(z)))
    for i in range(n):
        for j in range(i):
            o = int(orders[i, j])
            if o:
                m.AddBond(j, i, bond_type_dict[o])
    try:
        Chem.SanitizeMol(m.GetMol())
        return True
    except Exception:
        return False


@torch.inference_mode()
def evaluate(model, d: BondData, idx: torch.Tensor, device: str, batch: int,
             weights: torch.Tensor, n_validity: int):
    """CE, bond P/R/F1, exact-graph match, and molecule validity rate."""
    model.eval()
    ce_sum = n_pairs = 0.0
    tp = fp = fn = cls_ok = cls_tot = 0
    exact = n_mol = 0
    valid = checked = 0

    for s in range(0, idx.shape[0], batch):
        bidx = idx[s : s + batch]
        el, dist, adj_in, tgt, na = build_inputs(d, bidx, device)
        logits = model(elements=el, dist_mat=dist, adj_mat=adj_in)
        m = pair_mask(na, device)

        lg, tg = logits[m], tgt[m]
        ce_sum += F.cross_entropy(lg, tg, weight=weights, reduction="sum").item()
        n_pairs += tg.numel()

        pred = lg.argmax(-1)
        pb, tb = pred > 0, tg > 0
        tp += (pb & tb).sum().item()
        fp += (pb & ~tb).sum().item()
        fn += (~pb & tb).sum().item()
        cls_ok += (pred[tb] == tg[tb]).sum().item()
        cls_tot += int(tb.sum().item())

        full = logits.argmax(-1)
        wrong = ((full != tgt) & m).flatten(1).any(1)
        exact += int((~wrong).sum().item())
        n_mol += bidx.shape[0]

        if checked < n_validity:
            take = min(bidx.shape[0], n_validity - checked)
            orders = full.cpu().numpy()
            els = d.elements[bidx[:take]].numpy()
            nas = d.n_atoms[bidx[:take]].numpy()
            for k in range(take):
                valid += mol_is_valid(els[k], orders[k], int(nas[k]))
            checked += take

    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    return {
        "ce": ce_sum / max(n_pairs, 1),
        "bond_p": prec,
        "bond_r": rec,
        "bond_f1": 2 * prec * rec / max(prec + rec, 1e-9),
        "order_acc": cls_ok / max(cls_tot, 1),
        "exact": exact / max(n_mol, 1),
        "valid": valid / max(checked, 1),
        "n_valid_checked": checked,
    }


# --------------------- train ---------------------
d = BondData(DATA_DIR, MAX_SHARDS)
perm = torch.randperm(len(d), generator=torch.Generator().manual_seed(SEED))
n_val = max(1, int(len(d) * VAL_FRAC))
val_idx, train_idx = perm[:n_val], perm[n_val:]
log(f"split train={train_idx.shape[0]} val={val_idx.shape[0]}")

w = class_weights(d, 20_000, WEIGHT_CAP, device)
log("class weights " + " ".join(f"{k}:{v:.2f}" for k, v in enumerate(w.tolist())))

model = AdjMatSeer(n_hidden=N_HIDDEN, device=device).to(device)
n_par = sum(p.numel() for p in model.parameters())
log(f"AdjMatSeer n_hidden={N_HIDDEN}  params={n_par / 1e6:.2f} M ({n_par * 4 / 1e6:.1f} MB fp32)")
opt = AdamW(model.parameters(), lr=LR, weight_decay=1e-8)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

if BASELINE and Path(BASELINE).exists():
    base = AdjMatSeer(n_hidden=2048, device=device).to(device)
    bs = torch.load(BASELINE, map_location=device, weights_only=False)
    base.load_state_dict(bs.get("state_dict", bs))
    bm = evaluate(base, d, val_idx, device, BATCH, w, N_VALIDITY)
    log(
        f"BASELINE 2048  valid={bm['valid']:.3f}  <- the only comparable number "
        f"(README quotes 48% for this model); exact={bm['exact']:.3f} "
        f"bond_f1={bm['bond_f1']:.4f} order_acc={bm['order_acc']:.4f} are vs "
        f"{d.aromatic_mode} labels, so ignore them if mode=kekule"
    )
    del base
    if str(device).startswith("cuda"):
        torch.cuda.empty_cache()
elif BASELINE:
    log(f"baseline ckpt missing, skip: {BASELINE}")

best_valid, no_imp = -1.0, 0
for epoch in range(EPOCHS):
    model.train()
    ep = train_idx[torch.randperm(train_idx.shape[0])]
    run, ns = 0.0, 0
    pbar = tqdm(range(0, ep.shape[0] - BATCH + 1, BATCH), desc=f"h{N_HIDDEN} ep{epoch}")
    for s in pbar:
        bidx = ep[s : s + BATCH]
        el, dist, adj_in, tgt, na = build_inputs(d, bidx, device)
        logits = model(elements=el, dist_mat=dist, adj_mat=adj_in)
        m = pair_mask(na, device)
        loss = F.cross_entropy(logits[m], tgt[m], weight=w)
        if not torch.isfinite(loss):
            log(f"WARN non-finite loss ep={epoch}")
            continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        run += loss.item()
        ns += 1
        if ns % 50 == 0:
            pbar.set_postfix(ce=f"{run / ns:.4f}")
    sched.step()

    mt = evaluate(model, d, val_idx, device, BATCH, w, N_VALIDITY)
    log(
        f"ep={epoch} train_ce={run / max(ns, 1):.4f} val_ce={mt['ce']:.4f} "
        f"valid={mt['valid']:.3f} exact={mt['exact']:.3f} "
        f"bond_f1={mt['bond_f1']:.4f} bond_p={mt['bond_p']:.4f} bond_r={mt['bond_r']:.4f} "
        f"order_acc={mt['order_acc']:.4f}"
    )

    ckpt = {
        "state_dict": model.state_dict(),
        "n_hidden": N_HIDDEN,
        "epoch": epoch,
        "metrics": mt,
        "aromatic_mode": d.aromatic_mode,
        "config": {
            "data_dir": str(DATA_DIR), "out_dir": str(OUT_DIR),
            "n_hidden": N_HIDDEN, "epochs": EPOCHS, "batch": BATCH, "lr": LR,
            "baseline": str(BASELINE) if BASELINE else None,
        },
    }
    torch.save(ckpt, OUT_DIR / f"latest_seer_{N_HIDDEN}.pt")
    if mt["valid"] > best_valid:
        best_valid, no_imp = mt["valid"], 0
        torch.save(ckpt, OUT_DIR / f"best_seer_{N_HIDDEN}.pt")
        log(f"ckpt best valid={best_valid:.3f}")
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            log(f"early stop ep={epoch}")
            break

log(f"done n_hidden={N_HIDDEN} best_valid={best_valid:.3f}")